# Lab: ColBERT & Late Interaction RAG

In standard RAG, each text chunk is compressed into a single dense vector, so a chunk covering many topics gets blurred into one averaged representation. ColBERT takes a different approach: instead of one vector per chunk, it keeps **one vector per TOKEN**, preserving the fine-grained meaning of every word in the chunk.

When a query arrives, matching happens through **MaxSim (late interaction)**: every query token vector is compared against every document token vector, the maximum similarity per query token is taken, and those maxima are summed. This gives token-level precision — a chunk is rewarded for having individual tokens that strongly match specific query tokens, rather than just being broadly similar overall.

In this lab, we will:
1. Download the document (BERT paper).
2. Load the ColBERT model.
3. Create a Qdrant multivector collection for token-level vectors.
4. Run a late-interaction query.
5. Generate the final answer with an LLM.

In [2]:
!pip install -qU qdrant-client fastembed langchain-openai langchain-text-splitters pypdf requests ipython


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports

In [1]:
import uuid
import hashlib
import requests
from pypdf import PdfReader

# Vector Store
from qdrant_client import QdrantClient
from qdrant_client import models

# Embedding Model
from fastembed import LateInteractionTextEmbedding

# LLM & Text Processing
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Core Components
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Display
from IPython.display import Markdown, display

### Step 2: Configure Models

In [2]:
# Initialize Chat Model
llm = ChatOpenAI(
    openai_api_key="your-api-key",
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0.0
)


In [3]:
# ColBERT embeds every TOKEN, so the output is a (num_tokens, 128) matrix — not a single vector
colbert = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

### Step 3: Initialize Qdrant with Multivector Support or Use Existing Collection if Already Made

In [4]:
client = QdrantClient(
    url="your-endpoint", 
    api_key="your-api-key",
    timeout=300,
)
COLLECTION_NAME = "colbert_late_interaction"

# FIRST TIME RUN: creates a fresh collection (wipes & rebuilds it), then data is uploaded in Step 8
# ColBERT stores token MATRICES, so Qdrant needs a MULTIVECTOR collection
# compared with MaxSim instead of plain cosine similarity
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=128,
        distance=models.Distance.COSINE,
        multivector_config=models.MultiVectorConfig(
            comparator=models.MultiVectorComparator.MAX_SIM
        )
    )
)

C:\Users\risha\AppData\Local\Temp\ipykernel_40120\3870197828.py:11: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [22]:
# ALREADY RAN THIS NOTEBOOK? Uncomment this cell if you have already
# made the DB (collection) / uploaded data, then run it instead of Cell A
# (Cell A wipes and recreates the collection, so only use it the first time).
# client = QdrantClient(
#     url="your-endpoint", 
#     api_key="your-api-key",
#     timeout=300,
# )
# COLLECTION_NAME = "colbert_late_interaction"
# No recreate_collection here — the collection already exists.

C:\Users\risha\AppData\Local\Temp\ipykernel_35228\3984151268.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

### Step 4: Download & Extract Document

In [5]:
# BERT paper (source document for this lab)
PDF_URL = "https://arxiv.org/pdf/1810.04805.pdf"
PDF_FILENAME = "bert_paper.pdf"

# Download the PDF and save it locally; raise an error on HTTP failure
response = requests.get(PDF_URL)
response.raise_for_status()

with open(PDF_FILENAME, "wb") as f:
    f.write(response.content)

In [6]:
# Extract text from every page and join pages with newlines into one string
reader = PdfReader(PDF_FILENAME)
raw_text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

print(f"Extracted {len(raw_text)} characters.")

Extracted 64139 characters.


### Step 5: Chunk the Document

In [7]:
# Smaller chunks than Lab 1: ColBERT embeds every token, so big chunks would mean huge matrices
chunk_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = chunk_splitter.split_text(raw_text)

print(f"Created {len(chunks)} chunks.")

Created 144 chunks.


### Step 6: Generate Stable Chunk IDs

In [8]:
import uuid

# Deterministic ID from the chunk's own text, so the same chunk always gets the same ID across reruns
def stable_id(text):
    """Deterministic ID derived from the chunk text, so the same chunk always gets the same ID across runs."""
    # Use a fixed namespace (e.g., NAMESPACE_URL) to ensure stability
    return str(uuid.uuid5(uuid.NAMESPACE_URL, text))

# Same reasoning as Lab 1's Step 6 fix: stable payload IDs keep Qdrant consistent if this notebook is rerun
chunk_ids = [stable_id(chunk) for chunk in chunks]

### Step 7: Generate ColBERT Multi-Vector Embeddings

In [9]:
# Each chunk becomes a token matrix of shape (num_tokens, 128), not a single vector
chunk_embeddings = list(colbert.embed(chunks))

# Teaching sanity check: expect (num_tokens, 128)
print(f"First chunk embedding shape: {chunk_embeddings[0].shape}")

First chunk embedding shape: (93, 128)


### Step 8: Ingest Matrices into Qdrant

In [10]:
points = [
    models.PointStruct(
        id=stable_id(chunk),
        vector=embedding.tolist(),
        payload={"text": chunk},
    )
    for chunk, embedding in zip(chunks, chunk_embeddings)
]

# Upload in small batches (multivector points are big; large batches time out on free tier)
client.upload_points(collection_name=COLLECTION_NAME, points=points, batch_size=10)

print(f"Uploaded {len(points)} ColBERT points.")

Uploaded 144 ColBERT points.


### Step 9: Define the Late Interaction Query Function

In [11]:
def retrieve(query, top_k=3):
    """Embed the query as a token matrix and run a MaxSim (late interaction) search."""
    query_matrix = list(colbert.query_embed(query))[0]
    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_matrix.tolist(),
        limit=top_k,
    ).points
    return hits

### Step 10: Define the RAG Generation Chain

In [12]:
qa_template = """
You are a technical assistant. Answer the question using ONLY the provided context.

Context:
{context}

Question: {question}

Respond in exactly this format:

### Final Answer
<a clear, direct answer to the question>

### AI Tracing & Explainability
<Explain step by step how you arrived at the answer. Refer to the sources you used by their label (e.g. "Source 1"), state what each one contributed, and why it was relevant. Do not copy large blocks of text from the context — explain in your own words. Use as many lines as you need.>
"""

qa_prompt = ChatPromptTemplate.from_template(qa_template)

def format_hits(hits):
    """Label each retrieved chunk with its MaxSim score and payload text."""
    formatted = []
    for i, hit in enumerate(hits):
        formatted.append(f"[Source {i+1}] (MaxSim score: {hit.score})\n{hit.payload['text']}")
    return "\n\n".join(formatted)

# Prompt → LLM → plain string output
generation_chain = qa_prompt | llm | StrOutputParser()

### Step 11: Full Pipeline

In [13]:
def run_colbert_rag(query, top_k=3):
    """
    Runs the full pipeline: late-interaction retrieval, formatting, LLM generation.
    Displays the answer and returns both the hits and the answer.
    """
    hits = retrieve(query, top_k=top_k)
    context = format_hits(hits)
    answer = generation_chain.invoke({"context": context, "question": query})
    display(Markdown(answer))
    return hits, answer

### Step 12: Execute Query

In [15]:
query = "What pre-training tasks does BERT use to learn bidirectional representations?"
hits, answer = run_colbert_rag(query)

### Final Answer
BERT uses masked language models as its pre-training task to learn bidirectional representations.

### AI Tracing & Explainability
- **Source 1** explicitly states: "BERT uses masked language models to enable pre-trained deep bidirectional representations." This directly answers the question by naming the pre-training task (masked language models) and linking it to bidirectional representation learning.
- **Source 2** describes BERT's design goal (pre-training deep bidirectional representations by jointly conditioning on left and right context) but does not name the specific pre-training task.
- **Source 3** compares model architectures (BERT, OpenAI GPT, ELMo) and notes that only BERT is jointly conditioned on both left and right context in all layers, but it does not specify the pre-training task used.

Only Source 1 provides the name of the pre-training task, so the answer is based solely on that information.